# 🎯 Module 06: Clustering

> **Stage:** Machine Learning  
> **Level:** Intermediate  
> **Module:** 06 — Clustering  
> **Date:** 13 January 2026  
> **Previous:** 05 — Ensemble Methods  
> **Next:** 07 — Dimensionality Reduction

---

**Clustering** is the family of **unsupervised** algorithms that group similar data points together — **without labels**.

In this notebook we'll cover:

- What clustering is
- Distance & similarity
- **K-Means** and the **Elbow Method**
- **Hierarchical Clustering** and **dendrograms**
- **DBSCAN** — density-based clustering & noise
- **Gaussian Mixture Models (GMM)** — soft clustering
- **Silhouette Score**
- Common clustering problems and pitfalls

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what clustering is (and how it differs from supervised learning)
- Compute **Euclidean distance** between two points
- Apply **K-Means** and choose `K` with the **Elbow Method**
- Build a **hierarchical clustering** model and read a **dendrogram**
- Apply **DBSCAN** and tune `eps` / `min_samples`
- Fit a **Gaussian Mixture Model** and use **soft** cluster assignments
- Evaluate clusters with the **Silhouette Score**
- Diagnose the **wrong-scale** and **wrong-K** problems
- Interpret clusters in a **real-world** context

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)
print('✅ Setup ready.')

---
## 🧠 1. What is Clustering?

**Clustering** is an **unsupervised** ML technique that groups similar data points together.

Unlike supervised learning, there is **no target label**.

```text
Supervised:
X + y → Model → Prediction

Clustering:
X → Clustering Algorithm → Groups
```

### Example

```text
Customer Data
     ↓
Clustering
     ↓
┌──────────┐
│ Group 1  │ → High-value customers
│ Group 2  │ → Regular customers
│ Group 3  │ → Low-activity customers
└──────────┘
```

> The algorithm **discovers** groups; **you** interpret them.

---
## 📍 2. What Makes Data Points Similar?

Clustering algorithms need a way to measure **similarity** or **distance**.

### Euclidean distance

$$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$

For two points:

```text
A = (2, 3)
B = (5, 7)
```

$$d = \sqrt{(5-2)^2 + (7-3)^2} = \sqrt{9 + 16} = 5$$

Smaller distance → more similar points.

⚠️ **Distance-based algorithms are affected by feature scales** — scaling is often important before clustering.

In [ ]:
# Verify the Euclidean distance example
A = np.array([2, 3])
B = np.array([5, 7])
d = np.sqrt(((A - B) ** 2).sum())
print(f'Distance(A, B) = {d:.4f}')

---
## 🎯 3. K-Means Clustering

**K-Means** divides data into `K` clusters, each represented by a **centroid** (mean of its points).

### How K-Means works

```text
1. Choose K
2. Initialize cluster centroids
3. Assign each point to the nearest centroid
4. Recalculate centroids
5. Repeat until assignments stabilize
```

### Scikit-learn

```python
model = KMeans(n_clusters=3, random_state=42, n_init='auto')
labels = model.fit_predict(X)
```

In [ ]:
# Generate a synthetic 3-cluster dataset
X, y_true = make_blobs(
    n_samples=500, centers=3, cluster_std=1.2, random_state=42
)

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], s=25, alpha=0.7)
plt.title('Unlabeled data (3 natural groups)')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# K-Means in action
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
labels = kmeans.fit_predict(X)

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=25, alpha=0.7)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker='X', s=200, c='red', edgecolor='black', label='centroids')
plt.title('K-Means with K=3')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Inertia (within-cluster SSE): {kmeans.inertia_:.2f}')

---
## 📐 4. Choosing K — The Elbow Method

The **Elbow Method** compares clustering cost (inertia) for different `K`.

```text
Inertia
  │\
  │ \
  │  \
  │   \__
  │      \__
  └──────────── K
       ↑
     Elbow
```

The **"elbow"** is a useful candidate for `K` — not a guarantee.

In [ ]:
# Elbow method
ks = range(2, 11)
inertias = []

for k in ks:
    m = KMeans(n_clusters=k, random_state=42, n_init='auto')
    m.fit(X)
    inertias.append(m.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(ks), inertias, marker='o')
plt.xlabel('K'); plt.ylabel('Inertia')
plt.title('Elbow Method for K-Means')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🌳 5. Hierarchical Clustering

Hierarchical clustering builds a **hierarchy** of clusters.

### Agglomerative (bottom-up)

Starts with each point as its own cluster, then repeatedly merges the closest pairs:

```text
A  B  C  D  E
   ↓
A+B    C+D    E
   ↓
 A+B   C+D+E
   ↓
 A+B+C+D+E
```

### Visualized as a **dendrogram**

```text
       ┌─────────────┐
   ┌───┤             ├───┐
   A   B             C   D
```

Useful when the **relationships** between groups matter.

In [ ]:
# Dendrogram
Z = linkage(X[:50], method='ward')

plt.figure(figsize=(10, 4))
dendrogram(Z)
plt.title('Hierarchical Clustering — Dendrogram (first 50 points)')
plt.xlabel('Sample index'); plt.ylabel('Distance')
plt.tight_layout(); plt.show()

In [ ]:
# Agglomerative clustering
agg = AgglomerativeClustering(n_clusters=3)
agg_labels = agg.fit_predict(X)

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=agg_labels, cmap='viridis', s=25, alpha=0.7)
plt.title('Agglomerative Clustering (K=3)')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🌐 6. DBSCAN

**DBSCAN** = Density-Based Spatial Clustering of Applications with Noise.

Instead of requiring `K`, it finds **dense regions** and marks isolated points as **noise**.

### Parameters

- **`eps`** — maximum neighborhood distance
- **`min_samples`** — minimum neighbors required to form a dense region

### Strengths

- Finds **irregular-shaped** clusters
- Identifies **outliers** automatically

In [ ]:
# DBSCAN on crescent-shaped (non-spherical) data
X_moons, _ = make_moons(n_samples=400, noise=0.06, random_state=42)
X_moons_scaled = StandardScaler().fit_transform(X_moons)

db = DBSCAN(eps=0.3, min_samples=5)
db_labels = db.fit_predict(X_moons_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()

plt.figure(figsize=(6, 5))
plt.scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1],
            c=db_labels, cmap='viridis', s=25, alpha=0.8)
plt.title(f'DBSCAN on two-moons  |  clusters={n_clusters}  noise={n_noise}')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# DBSCAN on a spherical-cluster dataset for comparison
db2 = DBSCAN(eps=0.5, min_samples=5)
db2_labels = db2.fit_predict(X)

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=db2_labels, cmap='viridis', s=25, alpha=0.7)
plt.title('DBSCAN on 3 spherical clusters (may under-cluster due to uniform density)')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🧪 7. Gaussian Mixture Models (GMM)

A **GMM** assumes data comes from a mixture of several **Gaussian distributions**.

### Soft clustering

Instead of:

```text
Point → Cluster 1
```

we get probabilities:

```text
Point A:
  Cluster 1 → 0.80
  Cluster 2 → 0.20
```

### Comparison

| | K-Means | GMM |
| --- | --- | --- |
| Assignment | Hard | **Soft** |
| Shape | Spherical | Any Gaussian |
| Output | Label | Label + Probabilities |

In [ ]:
# GMM — labels and probabilities
gmm = GaussianMixture(n_components=3, random_state=42)
gmm.fit(X)
gmm_labels = gmm.predict(X)
gmm_probs  = gmm.predict_proba(X)

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=gmm_labels, cmap='viridis', s=25, alpha=0.7)
plt.title('GMM with 3 components')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('First 5 sample probabilities (soft assignment):')
print(gmm_probs[:5].round(3))

---
## 📊 8. Silhouette Score

How do we evaluate clustering when we have **no labels**?

The **Silhouette Score** measures how well each point fits its own cluster vs other clusters.

Range:

```text
-1 → Poor separation
 0 → Overlapping clusters
+1 → Strong separation
```

Higher is generally better — but **always** combine with domain knowledge and visualization.

In [ ]:
# Silhouette Score for various K on the blobs dataset
print(f'{"K":>3} {"Silhouette":>12}')
print('-' * 16)
for k in range(2, 8):
    m = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labs = m.fit_predict(X)
    score = silhouette_score(X, labs)
    print(f'{k:>3} {score:>12.4f}')

---
## ⚖️ 9. Comparing Clustering Algorithms

| Algorithm | Main Idea | Strength |
| --- | --- | --- |
| **K-Means** | Centroid-based | Simple and fast |
| **Hierarchical** | Cluster hierarchy | Shows relationships |
| **DBSCAN** | Density-based | Noise & irregular shapes |
| **GMM** | Probability distributions | Soft assignment |

### K-Means vs DBSCAN

| | K-Means | DBSCAN |
| --- | --- | --- |
| Number of clusters | Must choose `K` | Not required |
| Cluster shape | Compact spheres | Any shape |
| Outliers | Absorbed into clusters | Marked as noise |
| Main parameters | `K` | `eps`, `min_samples` |

In [ ]:
# Side-by-side comparison of 4 methods on the same data
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

algorithms = [
    ('K-Means (K=3)',         KMeans(n_clusters=3, random_state=42, n_init='auto').fit_predict(X)),
    ('Hierarchical (K=3)',    AgglomerativeClustering(n_clusters=3).fit_predict(X)),
    ('DBSCAN',                DBSCAN(eps=0.5, min_samples=5).fit_predict(X)),
    ('GMM (K=3)',             GaussianMixture(n_components=3, random_state=42).fit(X).predict(X)),
]

for ax, (name, labs) in zip(axes, algorithms):
    ax.scatter(X[:, 0], X[:, 1], c=labs, cmap='viridis', s=20, alpha=0.7)
    ax.set_title(name, fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle('Clustering Algorithms on the Same Data')
plt.tight_layout(); plt.show()

---
## 🛒 10. Real-World Example: Customer Segmentation

Imagine an e-commerce company has:

- Annual spending
- Purchase frequency
- Average order value

There are **no predefined groups**. We apply clustering:

```text
Customer Data
     ↓
Scale Features
     ↓
K-Means
     ↓
Cluster 0
Cluster 1
Cluster 2
```

Then we **interpret** what each cluster means: high-value, regular, low-activity, etc.

In [ ]:
# Simulate customer data
np.random.seed(42)
n = 600
spending     = np.concatenate([
    np.random.normal(2000,  500, 200),   # low-value
    np.random.normal(8000, 1000, 200),   # mid
    np.random.normal(20000, 2000, 200),  # high
])
frequency    = np.concatenate([
    np.random.normal(2, 0.5, 200),
    np.random.normal(5, 1.0, 200),
    np.random.normal(12, 1.5, 200),
])
avg_order    = np.concatenate([
    np.random.normal(80, 10, 200),
    np.random.normal(150, 20, 200),
    np.random.normal(250, 30, 200),
])

customers = pd.DataFrame({
    'annual_spending': spending,
    'purchase_freq':   frequency,
    'avg_order_value': avg_order,
})

print('Customer data:')
print(customers.head().round(1))

In [ ]:
# Scale, cluster, and profile the segments
scaler = StandardScaler()
X_cust = scaler.fit_transform(customers)

km = KMeans(n_clusters=3, random_state=42, n_init='auto')
customers['segment'] = km.fit_predict(X_cust)

profile = customers.groupby('segment').mean().round(1)
profile['count'] = customers.groupby('segment').size()
print('Cluster profiles:')
print(profile)

# Describe the segments in business language
profile = profile.sort_values('annual_spending')
labels = ['Low-activity customers',
          'Regular customers',
          'High-value customers']
print('\nInterpretation (sorted by annual spending):')
for lbl, (idx, row) in zip(labels, profile.iterrows()):
    print(f'  Segment {idx} → {lbl}')

In [ ]:
# Visualize the segments
plt.figure(figsize=(7, 5))
for seg in sorted(customers['segment'].unique()):
    sub = customers[customers['segment'] == seg]
    plt.scatter(sub['annual_spending'], sub['purchase_freq'],
                s=20, alpha=0.7, label=f'Segment {seg}')
plt.xlabel('Annual spending')
plt.ylabel('Purchase frequency')
plt.title('Customer segments (K-Means)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## ⚠️ 11. Common Clustering Problems

### Wrong Feature Scale

```text
Income = 500,000
Age    = 25
```

Income dominates distance → clusters ignore age.

**Solution:** Standardize features before clustering.

### Wrong Number of Clusters

For K-Means, poor `K` → meaningless groups.

**Solution:** Elbow method + Silhouette + domain knowledge.

### Assuming Every Cluster Has Business Meaning

A mathematically separated cluster isn't automatically a meaningful real-world segment. **Always inspect.**

In [ ]:
# Demonstrate the impact of not scaling
X_scale, _ = make_blobs(n_samples=500, centers=3, random_state=42)
X_scale[:, 0] *= 1000   # artificially blow up the first feature

labels_unscaled = KMeans(n_clusters=3, random_state=42, n_init='auto').fit_predict(X_scale)
labels_scaled   = KMeans(n_clusters=3, random_state=42, n_init='auto').fit_predict(
    StandardScaler().fit_transform(X_scale)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(X_scale[:, 0], X_scale[:, 1], c=labels_unscaled, cmap='viridis', s=20)
axes[0].set_title('Without scaling (first feature dominates)')
axes[1].scatter(X_scale[:, 0], X_scale[:, 1], c=labels_scaled, cmap='viridis', s=20)
axes[1].set_title('With StandardScaler')
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 📝 12. Practice Checklist

Use a customer dataset and:

- [ ] Select useful numerical features
- [ ] Scale the features
- [ ] Apply K-Means
- [ ] Test several values of `K`
- [ ] Use the Elbow Method
- [ ] Calculate Silhouette Score
- [ ] Try DBSCAN
- [ ] Try Hierarchical Clustering
- [ ] Try GMM
- [ ] Visualize the resulting clusters

---
## 🏋️ 13. Hands-On Exercises

### Exercise 1 — K sweep
Run K-Means with `K ∈ {2, 3, 4, 5, 6, 7}`. Plot Silhouette Score vs K. Which K is best?

### Exercise 2 — DBSCAN on Two-Moons
Try `eps ∈ {0.1, 0.2, 0.3, 0.5}` on the two-moons data. Which eps best recovers the two shapes?

### Exercise 3 — GMM Covariance Types
Try `covariance_type ∈ {'spherical', 'diag', 'tied', 'full'}`. Compare cluster shapes.

### Exercise 4 — Hierarchical Linkage
Try `method ∈ {'ward', 'complete', 'average', 'single'}`. How does the dendrogram change?

### Exercise 5 — Scaling Impact
Take a dataset with one feature on a much larger scale. Cluster with and without scaling. Compare Silhouette Scores.

### Exercise 6 — Real Dataset
Cluster the `load_wine()` dataset into 3 groups. Compare to the true labels with a contingency table.

---
## 🎤 14. Interview Questions

1. **What is clustering?**  
   An unsupervised technique that groups similar data points together without labels.

2. **Supervised vs unsupervised learning?**  
   Supervised uses labeled data; unsupervised finds structure in unlabeled data.

3. **How does K-Means work?**  
   It iteratively assigns points to the nearest centroid and updates centroids until convergence.

4. **What is a centroid?**  
   The mean position of all points in a cluster.

5. **How do you choose K?**  
   Elbow method, Silhouette Score, domain knowledge.

6. **What is the Elbow Method?**  
   Plotting inertia vs K and looking for the point where improvement slows down.

7. **What is Hierarchical Clustering?**  
   Building a hierarchy of clusters by repeatedly merging (or splitting) groups.

8. **What is a dendrogram?**  
   A tree-like diagram showing how clusters merge at each step.

9. **How does DBSCAN work?**  
   It groups points in regions with sufficient density and marks isolated points as noise.

10. **What are `eps` and `min_samples`?**  
    `eps` = maximum neighborhood radius; `min_samples` = minimum points required to form a dense region.

11. **How does DBSCAN handle outliers?**  
    It labels them as `-1` (noise) instead of forcing them into a cluster.

12. **What is a Gaussian Mixture Model?**  
    A probabilistic model that assumes data comes from a mixture of Gaussian distributions.

13. **Hard vs soft clustering?**  
    Hard → each point belongs to one cluster. Soft → each point has probabilities across clusters.

14. **What is the Silhouette Score?**  
    A measure of how well each point fits its cluster relative to others; range −1 to +1.

15. **Why scale features before clustering?**  
    Distance-based algorithms are dominated by large-scale features unless scaled.

---
## 🏁 15. Key Takeaways

```text
Clustering
     ↓
Unsupervised Learning
     ↓
Find groups in unlabeled data
```

### Main algorithms

```text
K-Means
   ↓
Centroid-based clusters

Hierarchical
   ↓
Cluster hierarchy

DBSCAN
   ↓
Density + noise detection

GMM
   ↓
Probability-based soft clustering
```

### The four mental models

> **K-Means = centroids**  
> **Hierarchical = hierarchy**  
> **DBSCAN = density + noise**  
> **GMM = probability distributions**

### 🧭 Golden Rule

> **Clustering finds structure — you assign meaning. Always scale features, validate with Silhouette + domain knowledge, and don't assume a cluster is inherently meaningful.**

---

### 🔗 What's Next?

**Module 07 — Dimensionality Reduction**

High-dimensional data is hard to visualize and often contains redundancy. We'll learn **PCA, t-SNE, UMAP, and LDA** to compress it while preserving structure.